Grundregeln:
- jede python Klasse hat eine übergeordnete Aufgabe
- jede python Klasse hält nur einen data file
- jede python Klasse importiert nur dann andere Klassen, wenn sie nur durch diese Importe als Gesamtes funktionieren (das ist in den meisten Fällen der DataManager). Wenn eine andere Klasse kein inhärenter Teil ist, wird sie nur als Funktions-Input für die Methode nutzbar gemacht.
- kein cleaning! Statt stillem Bereinigen stoppt die Funktion bei Ungereimtheiten und zeigt eine Fehlermeldung an

Zu adressierende Probleme:
- predicate requisite kleiner Teil der RO-Relationen. Reicht der aus?
- nur Abstracts in pubmed. Volltext-Reviews / Metaanalysen oder Bücher besser?
- manche Attribute sind bei fast jeder Entität TRUE. Sollte man diese entfernen? >90% TRUE?
- Scalability probleme vor allem der Normalisierung und fill_nan
- Vektordatenbank statt Pubmed_Abstracts_df besser?




Adressierte Probleme:
- LLM kosten sind explodiert, weil 120s von async gewartet wurde, die Antowrt brauchte aber 180s: geändert zu 240s Wartezeit
- LLM_ Input tokens sind explodiert wegen integration aller genutzten Targets und Attributes: Antwort: pruning auf die 300 meistgenutzten Targets
- causally_related ist oft falsch verknüpft. Aus der Liste der Prädikate entfernen? Antwort: Entfernt worden
- bis LLMClient().extract_entities werden nur Attribute ausgefüllt, die TRUE sind. Das ist zu ungenau. Von Anfang an müssen Attribute auch False gelabelt werden. Antwort: Extract entities labelt auch Falsche Entitäten
- mehrere Klassen haben durch das LLM denselben Namen bekommen, was verboten ist. Antwort: Prompt ist angepasst worden
- Abstracts enthalten sehr viele Entitäten, die nichts mit Wunden zu tun haben. Antwort: LLM prompt wurde so angepasst, dass nur domänenspezifische Entitäten integriert werden.
- Die formale Konzeptanalyse hält sich nicht an die BFO/OBO-Architektur. Antwort: attributes haben die form predicate: target. präformierte attributes sind die bfo_classes als is_a:bfo_class um die Einteilung in die BFo-Architektur zu erzwingen.
- es kann mehrere synonme Objekte oder Attribute geben. Antwort: objects und attributes werden per LLM zu standardbegriffen gemappt, dann grouping durch hard code
- Klassennamen werden aus Attributen gebildet. Antwort: Semantische Klassennamen kreiert LLM
- zu viele concepts und attribute? unschaffbar für hard code? erweiterter RAM und TPU
- zu viele Anfragen an LLM? Antwort: batching
- zu komplexe Anfragen an LLM? Antwort: Lange Frage besser als lange Antwort! (gemini 3 flash schafft 1mio tokens)
- komplette Nachverfolgbarkeit für selektives Löschen query-weise oder datumsweise bei schlechten einzelnen Ergebnissen: Antwort: meta_ids
- kein überladen des Speicher: Antwort: immer nur 10 dateien eines Präfix im Speicher und purge method des DataManager()
- LLM-Anfragen dauern zu lange, teils nur eine Anfrage pro 2,5 Minuten. Antwort: asyncio parallel prompting


# Setup

In [1]:
import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import sys
from pathlib import Path
from dotenv import load_dotenv

In [2]:
# Works whether VS Code runs the notebook from project root or notebooks/
_cwd = Path().resolve()
_project_root = _cwd if (_cwd / "config").exists() else _cwd.parent
sys.path.insert(0, str(_project_root))
load_dotenv(_project_root / ".env")

from config.settings import (
    BASE_PATH,
    OWL_OUTPUT_FILE,
    ENTREZ_EMAIL,
    GOOGLE_API_KEY,
    BATCH_SIZE,
    PUBMED_SEARCH_TERMS,
    ALLOWED_PREDICATES,
)
print(f"Project root: {_project_root}")
print(f"ALLOWED_PREDICATES: {ALLOWED_PREDICATES}")

Project root: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Forschung/Wound ontology
ALLOWED_PREDICATES: ['is_a', 'part_of', 'has_quality', 'occurs_in', 'caused_by', 'is_treatment_for']


In [3]:
from wound_ontology.orchestrator import Orchestrator

/opt/anaconda3/envs/first_env/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:648: ArbitraryTypeWarning: <property object at 0x11cae22f0> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(


# Pipeline

## 1. Initialise

In [4]:
pipeline = Orchestrator()

Directory ready: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Forschung/Wound ontology/data/cache
Loading latest JSON: meta_ids_2026-04-29_12-47.json
Initializing PubMed API...
Loading latest CSV: pubmed_abstracts_2026-04-27_18-38.csv
Connected to PubMed via moritz.milewski@master-bids.de
Initializing BFO Ontology...
No JSON found for prefix: bfo_class_labels
Loading BFO Ontology from http://purl.obolibrary.org/obo/bfo.owl...
Initializing Relations Ontology...
No JSON found for prefix: ro_property_labels
Loading Relations Ontology from http://purl.obolibrary.org/obo/ro.owl...
Initializing OBI Ontology...
No JSON found for prefix: obi_class_labels
Loading OBI Ontology from http://purl.obolibrary.org/obo/obi.owl...
Loading latest JSON: concept_store_2026-04-29_13-46.json
No CSV found for prefix: class_hierarchy
No CSV found for prefix: mereological_hierarchy
No CSV found for prefix: fca_hierarchy
No CSV found for prefix: context_matrix
No Pickle found for prefix: concep

## 2. Fetch abstracts

In [5]:
#pipeline.run_pubmed_search(PUBMED_SEARCH_TERMS)

## 3. Extract entities → ConceptStore

In [6]:
pipeline.scan_store_candidates() 

[StoreCleaner] Scoring 14920 unique entities via LLM (batches of 150)...
  Pre-blocked (in blocklist): 58
  Sending to LLM for evaluation: 14862
  LLM batch 1: 150/14862 evaluated, 121 flagged in this batch.
[LLM] Rate limited (429). Retry 1 in 60s...
[LLM] Rate limited (429). Retry 2 in 90s...
[LLM] Rate limited (429). Retry 3 in 120s...


KeyboardInterrupt: 

[LLM] Recovered after 3 retries.
  LLM batch 2: 300/14862 evaluated, 96 flagged in this batch.
  LLM batch 3: 450/14862 evaluated, 103 flagged in this batch.
  LLM batch 4: 600/14862 evaluated, 109 flagged in this batch.
  LLM batch 5: 750/14862 evaluated, 103 flagged in this batch.
[LLM] Rate limited (429). Retry 1 in 60s...
[LLM] Rate limited (429). Retry 2 in 90s...
[LLM] Recovered after 2 retries.
  LLM batch 6: 900/14862 evaluated, 75 flagged in this batch.
  LLM batch 7: 1050/14862 evaluated, 103 flagged in this batch.
  LLM batch 8: 1200/14862 evaluated, 100 flagged in this batch.
  LLM batch 9: 1350/14862 evaluated, 97 flagged in this batch.
  LLM batch 10: 1500/14862 evaluated, 101 flagged in this batch.
  LLM batch 11: 1650/14862 evaluated, 122 flagged in this batch.
  LLM batch 12: 1800/14862 evaluated, 103 flagged in this batch.
  LLM batch 13: 1950/14862 evaluated, 105 flagged in this batch.
  LLM batch 14: 2100/14862 evaluated, 43 flagged in this batch.
[LLM] Rate limited

In [ ]:
#pipeline.run_extract_to_store(batch_size=5, max_vocab_hints=300)

Using BFO anchor classes as is_a target candidates: ['material entity', 'immaterial continuant', 'quality', 'disposition', 'role', 'process']
Starting LLM Extraction...
Cleanup: Pruned old version meta_ids_2026-04-28_10-33.json
Parallel Extraction: 5377 new abstracts in waves of 5...
Cleanup: Pruned old version concept_store_2026-04-29_12-25.json
Cached PMID 41843965
Cached PMID 41924276
[LLM] Rate limited (429). Retry 1 in 60s...
Cached PMID 42010668
Cached PMID 41616704
Cleanup: Pruned old version concept_store_2026-04-29_12-26.json
Cached PMID 41586314
Cleanup: Pruned old version concept_store_2026-04-29_12-27.json
Cached PMID 41548164
Cleanup: Pruned old version concept_store_2026-04-29_12-28.json
Cached PMID 41537705
[LLM] Rate limited (429). Retry 1 in 60s...
Cached PMID 42042523
[LLM] Recovered after 1 retry.
Cached PMID 41583446
[LLM] Rate limited (429). Retry 1 in 60s...
Cached PMID 41537200
[LLM] Rate limited (429). Retry 1 in 60s...
Cleanup: Pruned old version concept_store_

KeyboardInterrupt: 

[LLM] Rate limited (429). Retry 1 in 60s...
Cleanup: Pruned old version concept_store_2026-04-29_13-30.json
Cached PMID 41846395
[LLM] Rate limited (429). Retry 2 in 90s...
[LLM] Rate limited (429). Retry 1 in 60s...
[LLM] Recovered after 3 retries.
Cached PMID 42010668
[LLM] Rate limited (429). Retry 1 in 60s...
Cleanup: Pruned old version concept_store_2026-04-29_13-31.json
Cached PMID 41805440
[LLM] Rate limited (429). Retry 1 in 60s...
Cached PMID 41819469
[LLM] Rate limited (429). Retry 2 in 90s...
[LLM] Rate limited (429). Retry 2 in 90s...
Cleanup: Pruned old version concept_store_2026-04-29_13-33.json
Cached PMID 41766349
[LLM] Rate limited (429). Retry 1 in 60s...
[LLM] Recovered after 2 retries.
Cached PMID 41795914
Cleanup: Pruned old version concept_store_2026-04-29_13-34.json
Cached PMID 41739511
[LLM] Rate limited (429). Retry 2 in 90s...
[LLM] Recovered after 1 retry.
Cached PMID 41775112
[LLM] Rate limited (429). Retry 3 in 120s...
Cleanup: Pruned old version concept_st

## 4. Build hierarchies (normalize + class / mereological)

In [ ]:
#pipeline.run_build_hierarchies()

## 5. Fill missing entity–attribute pairs

In [ ]:
#pipeline.run_store_fill()

## 6. Formal Concept Analysis

In [ ]:
#pipeline.run_fca(
    subtree_root="Wound",
    predicates=["has_quality", "occurs_in", "caused_by", "is_treatment_for"],
)

## 7. Export → OWL

In [ ]:
#pipeline.export_to_owl()

## 8. Enrich OWL (IRI renaming, BFO/RO import, UMLS synonyms)

In [ ]:
#pipeline.run_curate_owl(
    input_file=OWL_OUTPUT_FILE,
    output_file=str(Path(OWL_OUTPUT_FILE).parent / "ontology_enriched.owl"),
    import_bfo=True,
    import_ro=True,
    # umls_api_key=os.getenv("UMLS_API_KEY"),  # uncomment when key is available
)

# Inspection

In [ ]:
#pipeline.lattice.summary()

# Maintenance

> Run these cells selectively — they modify or delete cached data.

In [ ]:
# Purge a specific cache prefix and re-run from that step
# pipeline.dm.purge_by_prefix("context_matrix", confirm=True)